In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Unified Memory


> **Overview**
>
> 
>  * **Time:** 45 min
> 
>  1. Learn about Unified Memory in CUDA.
>  1. Understand how Unified Memory simplifies memory management in CUDA applications.

Unified Memory is a memory management feature in CUDA (CUDA 6 and above) that allows developers to write applications without
worrying about the complexities of managing memory between the host (CPU) and device (GPU).
It provides a single address space for both the host and device, enabling seamless data sharing and access.

In Unified Memory we use `cudaMallocManaged` to allocate memory that is accessible from both the host and device.

```cpp
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] = i;
}

// Use data on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

Under the hood, Unified Memory automatically migrates data between the host and device as needed.
This means that when the host accesses data that is currently on the device, Unified Memory will
automatically transfer it to the host memory, and vice versa. This migration is managed by the CUDA runtime,
which tracks memory accesses and performs the necessary transfers transparently.

The advantage of Unified Memory is:
* Simplifies programming — no need to manage explicit memory transfers
* Helps porting code from CPU to GPU
* Reduces the complexity of memory management

However, there are some considerations to keep in mind when using Unified Memory:

* May cause performance overhead due to page migration
* Less fine-grained control over memory movement
* Not all CUDA features are compatible with Unified Memory
* Limited support for certain data structures and algorithms

## Managed Memory and Pinned Host Memory


Managed memory is allocated with `cudaMallocManaged`. The runtime manages its accessibility and, where needed, migration between CPU and GPU. `cudaMemAdvise` provides hints for existing memory; it does not allocate memory.

Pinned host memory, allocated with `cudaMallocHost`, is a separate allocation type. It stays in host memory and is page-locked. The example below uses a separate device buffer and explicit copies; it does not rely on the GPU directly accessing a host pointer.

```cpp
// Example using pinned host memory and a separate device buffer
__global__ void kernel(int *data) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    data[idx] += 1; // Increment each element by 1
}

int main() {
    int size = 1024;
    int *data;

    // Allocate page-locked host memory and a separate GPU buffer
    cudaMallocHost(&data, size * sizeof(int));
    int *deviceData;
    cudaMalloc(&deviceData, size * sizeof(int));

    // Initialize data on the host
    for (int i = 0; i < size; i++) {
        data[i] = i;
    }

    cudaMemcpy(deviceData, data, size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch on device memory; size is exactly divisible by 256 here
    kernel<<<(size + 255) / 256, 256>>>(deviceData);

    // Synchronize to ensure all operations are complete
    cudaDeviceSynchronize();

    cudaMemcpy(data, deviceData, size * sizeof(int), cudaMemcpyDeviceToHost);
    cudaFree(deviceData);
    cudaFreeHost(data);
}
```

There are some disadvantages to using pinned memory:

* Pinned memory is not pageable, which means it cannot be swapped out to disk by the operating system.
* It can consume more system memory, as it is not eligible for paging.
* It may lead to reduced system performance if too much pinned memory is used, as it can limit the amount of memory available for other processes.

## cudaMemPrefetchAsync


The `cudaMemPrefetchAsync` function is used to prefetch data from the device to the host or vice versa in a
non-blocking manner. This can help improve performance by ensuring that data is available in the desired
memory space before it is accessed.

```cpp
// Example of using cudaMemPrefetchAsync
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Prefetch data to the host from GPU 0
cudaMemPrefetchAsync(data, size * sizeof(int), cudaCpuDeviceId);
cudaDeviceSynchronize();  // Wait for the prefetch before CPU access

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] += 1;
}

// Prefetch data back to GPU 0
cudaMemPrefetchAsync(data, size * sizeof(int), 0);

// Launch kernel on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

> **Explanation**
>
> 
>  `cudaCpuDeviceId` is a special device ID that refers to the host CPU.

## cudaMemAdvise


The `cudaMemAdvise` function is used to provide advice to the CUDA runtime about how memory should be managed.

```cpp
// Example of using cudaMemAdvise
int *data;
cudaMallocManaged(&data, N * sizeof(int));

// Initialize data on host
for (int i = 0; i < N; ++i)
    data[i] = i;

// 1. Advise that the data is mostly read, with occasional writes
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetReadMostly, cudaCpuDeviceId);

// 2. Prefer memory to be located on GPU 0
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetPreferredLocation, 0);

// 3. Specify that GPU 0 will access this memory
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetAccessedBy, 0);
```

The different advices that can be provided using `cudaMemAdvise` include:

* `cudaMemAdviseSetReadMostly`: Indicates mostly reads with occasional writes; it is not a CPU-only hint.
* `cudaMemAdviseSetPreferredLocation`: Specifies the preferred location for the memory (host or device).
* `cudaMemAdviseSetAccessedBy`: Indicates which device(s) will access the memory.

```cpp
// Example of using cudaMemAdvise with different advices
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Advise that the data is mostly read, with occasional writes
cudaMemAdvise(data, size * sizeof(int), cudaMemAdviseSetReadMostly, 0);

// Advise the CUDA runtime that the data will be accessed by device 0
cudaMemAdvise(data, size * sizeof(int), cudaMemAdviseSetAccessedBy, 0);

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] += 1;
}

// Launch kernel on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

> **Key Points**
>
> 
>  1. Unified Memory provides a single address space for both host and device memory.
>  1. It simplifies memory management by automatically migrating data between host and device.
>  1. Use `cudaMallocManaged` to allocate Unified Memory.
>  1. Be aware of potential performance overhead due to automatic page migration.
>  1. Not all CUDA features are compatible with Unified Memory, so check compatibility when using it.
>  1. Pinned memory can improve performance for certain operations but requires careful management.
>  1. Use `cudaMemPrefetchAsync` to prefetch data between host and device in a non-blocking manner.
>  1. Use `cudaMemAdvise` to provide advice to the CUDA runtime about memory management.

## Notes on the examples

The notebook clarifies the source's distinction between managed allocations and pinned host allocations, corrects the role of `cudaMemAdvise`, and waits for host prefetch completion before CPU access. See [NVIDIA CUDA 12.9 memory management](https://docs.nvidia.com/cuda/archive/12.9.0/cuda-runtime-api/group__CUDART__MEMORY.html).

The tutorial snippets illustrate API usage and omit some setup, error handling, or cleanup. The exercise below uses a complete program. Its prefetch calls follow the CUDA 12.9 API used by this workshop.

## Exercise 09: Unified Memory

**Time: 20 min**

Compare the supplied program before and after enabling its GPU-prefetch calls.
[Exercise instructions](../docs/source/tutorial/exercises/09-unified-memory.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/09-unified-memory/vector-add-unified.cu" target="_blank" rel="noopener noreferrer">Open vector-add-unified.cu</a>

### Baseline: prefetch disabled

Check the three GPU-prefetch calls are commented out, then submit the prebuilt
baseline. If source changed, first rebuild in `02-gpu.ipynb`.

In [ ]:
# Submit a new job, then inspect the queue.
baseline_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 09-unified-memory.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
baseline_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if baseline_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
show_job_output("09-unified-memory", baseline_job_id)

### Enable GPU prefetching

After the baseline job finishes, edit `exercises/09-unified-memory/vector-add-unified.cu`.
Uncomment the GPU-prefetch calls for `A`, `B`, and `C`. Rebuild using the build cell
in `02-gpu.ipynb`, then submit the same documented job again. Do not rebuild while
the baseline job is running. No temporary variant or compilation cell is used here.

In [ ]:
# Submit a new job, then inspect the queue.
prefetch_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 09-unified-memory.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
prefetch_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if prefetch_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


In [ ]:
show_job_output("09-unified-memory", prefetch_job_id)

### Compare

Both runs must report `PASSED`. Keep their separate PBS logs and record the
reported times. CPU verification and its managed-memory accesses happen after
the timed interval. Why might prefetching change performance? Is it consistently
faster? Use the same source workload and timing boundaries for both runs.

| Run | No GPU prefetch (ms) | GPU prefetch enabled (ms) |
| --- | --- | --- |
| 1 | | |
| 2 | | |
| 3 | | |

## Next: final image-blur project

Continue with [Final Project: Image Blur](../docs/source/tutorial/exercises/00-project.rst)
after this exercise. Its skeleton combines shared memory, asynchronous operations,
events and unified memory. Complete its TODOs before expecting GPU checks to pass.